# Pretrained Fake News Detector Audit

This notebook applies a pre-trained fake news detector model to rewritten texts.

Flow:
1) Loads rewritten datasets
2) Runs inference on rewritten text
3) (Optional) Compares against real labels, if available
4) Saves per-dataset and consolidated results

## 1) Imports

In [ ]:
import os
import time
from pathlib import Path

import pandas as pd
from sklearn.metrics import classification_report
from transformers import AutoModelForSequenceClassification, AutoTokenizer

## 2) Configuration

In [ ]:
# Folder with rewritten CSVs exported by the simulation notebook
run_id_env = os.getenv("RUN_ID")
if run_id_env:
    RUN_ID = run_id_env
    INPUT_DIR = Path("../output/runs") / RUN_ID / "rewritten"
    AUDIT_OUTPUT_DIR = Path("../output/runs") / RUN_ID / "audit" / "PreTrainedAudit"
else:
    RUN_ID = None
    INPUT_DIR = Path("../output/rewritten")
    AUDIT_OUTPUT_DIR = Path("../output/audit/PreTrainedAudit")

# Global dataset selection:
# - "ALL" to process all CSVs in INPUT_DIR
# - file name (e.g.: "science_rewritten_df.csv") to process a single file
DATASET_SELECTOR = "ALL"

# Text column used for inference
REWRITTEN_COLUMN = "rewritten_news"

# Real label column (optional). If None, supervised classification is skipped
LABEL_COLUMN = None  # e.g.: "label"

# Pre-trained fake news detector
DETECTOR_MODEL = "jy46604790/Fake-News-Bert-Detect"

MAX_ROWS_PER_DATASET = 1000

# Output folder
AUDIT_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("Configuration loaded.")
if RUN_ID:
    print(f"RUN_ID: {RUN_ID}")
else:
    print("RUN_ID not set. Using default folders in ../output.")
print(f"INPUT_DIR: {INPUT_DIR}")
print(f"DATASET_SELECTOR: {DATASET_SELECTOR}")
print(f"REWRITTEN_COLUMN: {REWRITTEN_COLUMN}")
print(f"DETECTOR_MODEL: {DETECTOR_MODEL}")
print(f"AUDIT_OUTPUT_DIR: {AUDIT_OUTPUT_DIR}")

## 3) Utility functions

In [ ]:
from utils.bert_audit_functions import (
    pretrained_fake_news_detector_prediction,
    read_dataset,
)
from utils.run_report import append_execution_report, resolve_execution_report_path

## 4) Load datasets

In [ ]:
if not INPUT_DIR.exists():
    raise FileNotFoundError(f"Input folder not found: {INPUT_DIR}")

if DATASET_SELECTOR.strip().upper() == "ALL":
    selected_files = sorted(INPUT_DIR.glob("*.csv"), key=lambda p: p.name.lower())
else:
    selected_file = INPUT_DIR / DATASET_SELECTOR
    if not selected_file.exists():
        raise FileNotFoundError(f"Selected file not found: {selected_file}")
    selected_files = [selected_file]

if not selected_files:
    raise FileNotFoundError(f"No CSV found in: {INPUT_DIR}")

dataset_payloads = []
for file_path in selected_files:
    df = read_dataset(file_path).copy()

    if REWRITTEN_COLUMN not in df.columns:
        raise ValueError(f"Text column not found in {file_path.name}: {REWRITTEN_COLUMN}")

    df[REWRITTEN_COLUMN] = df[REWRITTEN_COLUMN].fillna("").astype(str).str.strip()
    df = df[df[REWRITTEN_COLUMN] != ""].copy()
    df = df.head(MAX_ROWS_PER_DATASET).copy()

    dataset_payloads.append(
        {
            "file_path": file_path,
            "file_name": file_path.name,
            "dataset_name": file_path.stem,
            "df": df,
        }
    )

print(f"Selected datasets: {len(dataset_payloads)}")
for payload in dataset_payloads:
    print(f"- {payload['file_name']}: {len(payload['df'])} rows")

dataset_payloads[0]["df"][[REWRITTEN_COLUMN]].head()

## 5) Load detector and run inference

In [ ]:
detector_started_at = time.perf_counter()
tokenizer = AutoTokenizer.from_pretrained(DETECTOR_MODEL)
model = AutoModelForSequenceClassification.from_pretrained(DETECTOR_MODEL)
model.eval()

all_predictions = []
summary_rows = []

for payload in dataset_payloads:
    current_df = payload["df"].copy()
    prediction_rows = []

    for idx, text in current_df[REWRITTEN_COLUMN].items():
        pred = pretrained_fake_news_detector_prediction(tokenizer, model, text)
        prediction_rows.append(
            {
                "row_index": int(idx),
                "prediction_id": pred["prediction_id"],
                "prediction_label": pred["prediction_label"],
                "prediction_confidence": pred["prediction_confidence"],
            }
        )

    pred_df = pd.DataFrame(prediction_rows)
    result_df = current_df.join(pred_df.set_index("row_index"), how="left")
    result_df["source_file"] = payload["file_name"]
    result_df["dataset_name"] = payload["dataset_name"]

    all_predictions.append(result_df)

    fake_rate = (
        result_df["prediction_label"].str.lower().eq("fake").mean() if len(result_df) > 0 else 0.0
    )
    summary_rows.append(
        {
            "dataset_name": payload["dataset_name"],
            "source_file": payload["file_name"],
            "rows": int(len(result_df)),
            "fake_rate": float(fake_rate),
        }
    )

predictions_all_df = (
    pd.concat(all_predictions, ignore_index=True) if all_predictions else pd.DataFrame()
)
summary_df = pd.DataFrame(summary_rows).sort_values("fake_rate", ascending=False)
detector_elapsed_seconds = time.perf_counter() - detector_started_at

print("Summary by dataset:")
display(summary_df)

predictions_all_df[
    ["dataset_name", REWRITTEN_COLUMN, "prediction_label", "prediction_confidence"]
].head(10)

## 6) (Optional) Classification report

In [ ]:
if LABEL_COLUMN and LABEL_COLUMN in predictions_all_df.columns:
    y_true = predictions_all_df[LABEL_COLUMN].astype(str)
    y_pred = predictions_all_df["prediction_label"].astype(str)
    print(classification_report(y_true, y_pred, zero_division=0))
else:
    print("LABEL_COLUMN not defined/found. Skipping classification_report.")

## 7) Save results

In [ ]:
if predictions_all_df.empty:
    raise ValueError("No result was generated.")

per_file_outputs = []
for dataset_name, dataset_df in predictions_all_df.groupby("dataset_name", dropna=False):
    output_file = AUDIT_OUTPUT_DIR / f"{dataset_name}_pretrained_fake_news_predictions.csv"
    dataset_df.to_csv(output_file, index=False)
    per_file_outputs.append(output_file)

combined_output = AUDIT_OUTPUT_DIR / "all_datasets_pretrained_fake_news_predictions.csv"
predictions_all_df.to_csv(combined_output, index=False)

summary_output = AUDIT_OUTPUT_DIR / "pretrained_fake_news_summary.csv"
summary_df.to_csv(summary_output, index=False)

print("Saved files:")
for output_file in per_file_outputs:
    print(f"- {output_file}")
print(f"- {combined_output}")
print(f"- {summary_output}")

run_id_for_report, report_path = resolve_execution_report_path()

detector_dataset_metrics = []
if not summary_df.empty:
    for row in summary_df.to_dict("records"):
        detector_dataset_metrics.append(
            {
                "dataset_name": row.get("dataset_name"),
                "source_file": row.get("source_file"),
                "rows_audited": int(row.get("rows", 0) or 0),
                "fake_count": int(row.get("fake_count", 0) or 0),
                "fake_rate": float(row.get("fake_rate", 0.0) or 0.0),
            }
        )

fake_count_total = int((predictions_all_df["prediction_label"] == "FAKE").sum())
fake_rate_total = (
    (fake_count_total / len(predictions_all_df)) if len(predictions_all_df) > 0 else 0.0
)

append_execution_report(
    report_path=report_path,
    notebook_name="pretrained_fake_news_detector_workbench.ipynb",
    section_title="Pretrained fake-news detector execution",
    run_id=run_id_for_report,
    details={
        "rows_audited": int(len(predictions_all_df)),
        "detector_execution_seconds": round(detector_elapsed_seconds, 3),
        "pretrained_auditor_used": DETECTOR_MODEL,
        "total_dataset_rows": int(sum(len(payload["df"]) for payload in dataset_payloads)),
        "audit_model_metrics": [
            {
                "audit_model": DETECTOR_MODEL,
                "fake_count": fake_count_total,
                "fake_rate": round(fake_rate_total, 6),
            }
        ],
        "detector_dataset_metrics": detector_dataset_metrics,
        "execution_details": {
            "dataset_selector": DATASET_SELECTOR,
            "input_dir": str(INPUT_DIR),
            "rewritten_column": REWRITTEN_COLUMN,
        },
    },
)

print(f"Execution report updated: {report_path}")